## Setup

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata

folderPath = userdata.get("reinforcementLearningFolderPath")

In [3]:
roms_path = f"{folderPath}/Project/roms"

!mkdir -p roms
!cp -r "$roms_path" ./

In [4]:
!pip install -q stable-retro coloredlogs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.6 MB/s eta 0:00:00


In [5]:
!python -m retro.import ./roms

Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [7]:
import coloredlogs
import logging
from datetime import datetime
from logging import Logger, FileHandler, Formatter, LoggerAdapter

def get_logger(run_name: str, level: str):
    logger = logging.getLogger(run_name)
    logger.setLevel(level)
    coloredlogs.install(level=level, logger=logger, fmt="%(asctime)s [%(levelname)s] %(message)s", isatty=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file_name = f"{run_name}_{timestamp}.log"
    file_handler = FileHandler(log_file_name)
    file_handler.setLevel(level)
    formatter = Formatter("%(asctime)s [%(levelname)s] %(message)s")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.info(f"Session log for run {run_name} with level [{level}] initialized at: {log_file_name}")
    return logger

class DummyLogger(Logger):
    def __init__(self):
        super().__init__("dummy")
    def debug(self, msg, *args, **kwargs):
        pass
    def info(self, msg, *args, **kwargs):
        pass
    def warning(self, msg, *args, **kwargs):
        pass
    def error(self, msg, *args, **kwargs):
        pass
    def critical(self, msg, *args, **kwargs):
        pass
    def exception(self, msg, *args, **kwargs):
        pass

# Initial Testing

In [8]:
import json

def update_emulator_setup(setup: dict):
    path = '/usr/local/lib/python3.12/dist-packages/stable_retro/data/stable/SuperMarioWorld-Snes-v0/data.json'
    with open(path, 'w') as f:
        json.dump(setup, f, indent=2)

In [9]:
emulator_setup = {
  "info": {
    "score":        {"address": 8261428, "type": "<u4"},
    "lives":        {"address": 8261054, "type": "|i1"},
    "x":            {"address": 0x94, "type": "<u2"},  # Mario X position (16-bit)
    "y":            {"address": 0x96, "type": "<u2"},  # Mario Y position (16-bit)
    "cam_x":        {"address": 0x1A,   "type": "<u2"}, # $7E:001A (Camera Scroll X)
    "cam_y":        {"address": 0x1C,   "type": "<u2"}, # $7E:001C (Camera Scroll Y)
    "mario_powerup": {"address": 0x0019, "type": "|u1"}, # 0=Small, 1=Big, 2=Cape, 3=Fire
    "mario_climbing": {"address": 0x0074, "type": "|u1"}, # Non-zero if on a vine/ladder
    "mario_y_screen": {"address": 0x00D3, "type": "|u1"}, # Vertical screen Mario is on
    "mario_priority": {"address": 0x13F9, "type": "|u1"}, # Layering priority for Mario
    "item_in_hand": {"address": 0x0DC2, "type": "|u1"}, #carrying key
    "end_level_timer": {"address": 0x1493, "type": "|u1"}, # $7E:1493 non-zero when level ends (either exit)
    "room_id":        {"address": 0x010B, "type": "|u1"}, # $7E:010B current sublevel/room number
    "game_mode":      {"address": 0x0100, "type": "|u1"}, # $7E:0100 game mode (0x14 = in level, 0x0B = overworld)

  }
}

for i in range(12):
    emulator_setup["info"][f"sprite_status_{i}"] = {"address": 0x14C8 + i, "type": "|u1"} # 0 = Empty, 8 = Active
    emulator_setup["info"][f"sprite_type_{i}"]   = {"address": 0x009E + i, "type": "|u1"} # What is it? (Goomba, Koopa, etc.)
    emulator_setup["info"][f"sprite_x_low_{i}"]  = {"address": 0x00E4 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_x_high_{i}"] = {"address": 0x14E0 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_y_low_{i}"]  = {"address": 0x00D8 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_y_high_{i}"] = {"address": 0x14D4 + i, "type": "|u1"}
    emulator_setup["info"][f"sprite_priority_{i}"] = {"address": 0x15F6 + i, "type": "|u1"}

for i in range(10):
    emulator_setup["info"][f"ext_sprite_type_{i}"]   = {"address": 0x170B + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_x_low_{i}"]  = {"address": 0x171F + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_x_high_{i}"] = {"address": 0x1733 + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_y_low_{i}"]  = {"address": 0x1715 + i, "type": "|u1"}
    emulator_setup["info"][f"ext_sprite_y_high_{i}"] = {"address": 0x1729 + i, "type": "|u1"}

WRAM_BASE = 8257536
TILE_LAYER_1_LOW = 0xC800
OFFSET_TILE_LAYER_1_LOW = TILE_LAYER_1_LOW + WRAM_BASE

for i in range(512):
    emulator_setup["info"][f"tile_layer1_low_{i}"] = {"address": OFFSET_TILE_LAYER_1_LOW + i, "type": "|u1"}

update_emulator_setup(emulator_setup)

In [10]:
import enum
from typing import TypedDict, Optional

class TileType(enum.Enum):
    EMPTY = 0
    MARIO = 1
    SPRITE = 2
    EXTENDED_SPRITE = 3
    TERRAIN = 4
    BLOCK = 5

class Tile(TypedDict):
    type: TileType
    id: int

def tile_absolute_id(tile: Tile) -> int:
    if tile["type"] == TileType.EMPTY:
        return 0
    if tile["type"] == TileType.MARIO:
        return 1
    offset_level = 0
    offset_multiplier = 256
    if tile["type"] == TileType.SPRITE:
        offset_level = 1
    if tile["type"] == TileType.EXTENDED_SPRITE:
        offset_level = 2
    if tile["type"] == TileType.TERRAIN:
        offset_level = 3
    if tile["type"] == TileType.BLOCK:
        offset_level = 4
    offset = offset_level * offset_multiplier
    return int(tile["id"]) + offset

class LayeredTile():
    def __init__(self, type: TileType, id: int, layer: int):
        self.type = type
        self.id = id
        self.layer = layer

    def get_tile(self) -> Tile:
        return Tile(type=self.type, id=self.id)

class UngriddedTile(LayeredTile):
    def __init__(self, type: TileType, id: int, layer: int, x: int, y: int):
        super().__init__(type, id, layer)
        self.x = x
        self.y = y

class TileStack():
    def __init__(self):
        self.stack: list[LayeredTile] = []

    def add(self, tile: LayeredTile):
        self.stack.append(tile)

    def resolve(self) -> Tile:
        if len(self.stack) == 0:
            return Tile(type=TileType.EMPTY, id=0)
        front_tile = self.stack[0]
        for tile in self.stack:
            if tile.layer < front_tile.layer:
                front_tile = tile
        return front_tile.get_tile()

class DenseTileMatrix():
    def __init__(self, rows: int = 14, columns: int = 16):
        self.rows = rows
        self.columns = columns
        self.matrix: list[list[TileStack]] = []
        for row in range(rows):
            row_stack = []
            for col in range(columns):
                row_stack.append(TileStack())
            self.matrix.append(row_stack)

    def add(self, tile: Optional[UngriddedTile]):
        if tile is None:
            return
        if not self._are_grid_coords_in_cam_view(tile.x, tile.y):
            return
        self.matrix[tile.y][tile.x].add(tile)

    def _are_grid_coords_in_cam_view(self, grid_x: int, grid_y: int) -> bool:
        return 0 <= grid_x < self.columns and 0 <= grid_y < self.rows

    def resolve(self):
        resolved_matrix = []
        for row in self.matrix:
            resolved_row = []
            for col in row:
                resolved_row.append(col.resolve())
            resolved_matrix.append(resolved_row)
        return resolved_matrix

    def resolveSimplified(self):
        resolved_matrix = []
        for row in self.matrix:
            resolved_row = []
            for col in row:
                resolved_row.append(tile_absolute_id(col.resolve()))
            resolved_matrix.append(resolved_row)
        return np.array(resolved_matrix)

In [11]:
from typing import Optional

import numpy as np

class WorldParser:
    def __init__(self, env, logger: Optional[Logger] = None):
        self.logger: Logger = logger if logger is not None else DummyLogger()
        self.env = env
        self.SCREEN_ROWS = 14
        self.SCREEN_COLUMNS = 16
        self.TILE_SIZE = 16

    def get_screen_matrix(self, info) -> np.matrix:
        screen_matrix = self._build_dense_screen_matrix(info)
        return screen_matrix.resolve()

    def get_screen_matrix_simplified(self, info) -> np.matrix:
        screen_matrix = self._build_dense_screen_matrix(info)
        return screen_matrix.resolveSimplified()

    def _build_dense_screen_matrix(self, info) -> DenseTileMatrix:
        screen_matrix = DenseTileMatrix()
        for mario_tile in self._get_mario(info):
            screen_matrix.add(mario_tile)
        for sprite in self._get_sprites(info):
            screen_matrix.add(sprite)
        for extended_sprite in self._get_extended_sprites(info):
            screen_matrix.add(extended_sprite)
        for tile in self._get_layer1_tiles(info):
            screen_matrix.add(tile)
        return screen_matrix

    def _get_mario(self, info: dict) -> list[UngriddedTile]:
        mario_tiles = []
        mario_x = info.get("x", 0)
        mario_y = info.get("y", 0)
        grid_x, grid_y = self._coords_absolute_to_cam_grid(mario_x, mario_y, info)
        grid_y += 1 # Fixing shift in mario position
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning("Mario out of bounds")
            return []
        mario_priority = info.get("mario_priority", 2)
        layer = 1 if mario_priority == 3 else 3
        mario_tiles.append(UngriddedTile(type=TileType.MARIO, id=1, layer=layer, x=grid_x, y=grid_y))
        is_big = info.get('mario_powerup', 0) > 0
        if is_big and grid_y > 0:
            mario_tiles.append(UngriddedTile(type=TileType.MARIO, id=2, layer=layer, x=grid_x, y=grid_y - 1))
        return mario_tiles

    def _get_sprites(self, info: dict) -> list[UngriddedTile]:
        sprites = []
        for sprite_index in range(12):
            sprite = self._get_sprite_at_index(sprite_index, info)
            if sprite is not None:
                sprites.append(sprite)
        return sprites

    def _get_sprite_at_index(self, index: int, info: dict) -> Optional[UngriddedTile]:
        if info.get(f"sprite_status_{index}", 0) == 0:
            return None
        sprite_x = self._combine_low_and_high_bits(info.get(f"sprite_x_low_{index}", 0), info.get(f"sprite_x_high_{index}", 0))
        sprite_y = self._combine_low_and_high_bits(info.get(f"sprite_y_low_{index}", 0), info.get(f"sprite_y_high_{index}", 0))
        sprite_type = info.get(f"sprite_type_{index}", 0)
        grid_x, grid_y = self._coords_absolute_to_cam_grid(sprite_x, sprite_y, info)
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning(f"Sprite {index} out of bounds")
            return None
        priority_byte = info.get(f"sprite_priority_{index}", 0)
        sprite_priority = priority_byte & 0x03
        layer_map = {3: 1, 2: 3, 1: 5, 0: 6}
        layer = layer_map.get(sprite_priority, 3)
        return UngriddedTile(type=TileType.SPRITE, id=sprite_type, layer=layer, x=grid_x, y=grid_y)

    def _get_extended_sprites(self, info: dict) -> list[UngriddedTile]:
        extended_sprites = []
        for sprite_index in range(10):
            extended_sprite = self._get_extended_sprite_at_index(sprite_index, info)
            if extended_sprite is not None:
                extended_sprites.append(extended_sprite)
        return extended_sprites

    def _get_extended_sprite_at_index(self, index: int, info: dict) -> Optional[UngriddedTile]:
        sprite_type = info.get(f"ext_sprite_type_{index}", 0)
        if sprite_type == 0:
            return None
        sprite_x = self._combine_low_and_high_bits(info.get(f"ext_sprite_x_low_{index}", 0), info.get(f"ext_sprite_x_high_{index}", 0))
        sprite_y = self._combine_low_and_high_bits(info.get(f"ext_sprite_y_low_{index}", 0), info.get(f"ext_sprite_y_high_{index}", 0))
        grid_x, grid_y = self._coords_absolute_to_cam_grid(sprite_x, sprite_y, info)
        grid_x += 1
        if not self._are_grid_coords_in_cam_view(grid_x, grid_y):
            # self.logger.warning(f"Extended sprite {index} out of bounds")
            return None
        return UngriddedTile(type=TileType.EXTENDED_SPRITE, id=sprite_type+200, layer=3, x=grid_x, y=grid_y)

    def _get_layer1_tiles(self, info: dict) -> list[UngriddedTile]:
        ram = self.env.unwrapped.get_ram()
        RAM_BASE_OFFSET = 0xF000
        ATTR_BASE_OFFSET = RAM_BASE_OFFSET + 0x10000
        cam_x, cam_y = self._get_camera_coords(info)
        tiles = []

        # Iterate through the 14x16 view of the camera
        for row in range(self.SCREEN_ROWS):
            for col in range(self.SCREEN_COLUMNS):
                # Calculate world pixel coordinates
                world_x = cam_x + (col * 16)
                world_y = cam_y + (row * 16)

                # 1. Convert pixels to tile coordinates
                tx = world_x // 16
                ty = world_y // 16

                # 2. Safety check: SMW levels are 27 tiles high (0 to 26)
                if ty < 0 or ty >= 27:
                    continue

                # 3. Determine which screen (16-tile wide chunk) we are in
                screen_num = tx // 16

                # 4. Local X within that screen (0 to 15)
                lx = tx % 16

                # 5. Calculate the index based on the 27-row height (0x1B0 bytes per screen)
                # Index = (Screen * TilesPerScreen) + (Row * TilesPerRow) + Column
                tile_index = (screen_num * 432) + (ty * 16) + lx

                # 6. Safety check for the Map16 table limit (14336 bytes)
                if tile_index < 14336:
                    tile_id = ram[RAM_BASE_OFFSET + tile_index]
                    if tile_id != 0x25: # Sky tiles
                        attr_byte = ram[ATTR_BASE_OFFSET + tile_index]
                        has_priority = (attr_byte & 0x20) != 0
                        layer = 2 if has_priority else 4
                        tiles.append(UngriddedTile(
                            type=TileType.TERRAIN,
                            id=tile_id,
                            layer=layer,
                            x=col,
                            y=row
                        ))
        return tiles

    def _combine_low_and_high_bits(self, low: int, high: int) -> int:
        return (high << 8) | low

    def _are_grid_coords_in_cam_view(self, grid_x: int, grid_y: int) -> bool:
        return 0 <= grid_x < self.SCREEN_COLUMNS and 0 <= grid_y < self.SCREEN_ROWS

    def _coords_absolute_to_cam_grid(self, x: int, y: int, info: dict) -> tuple[int, int]:
        relative_x, relative_y = self._coords_absolute_to_cam_relative(x, y, info)
        grid_x, grid_y = relative_x // self.TILE_SIZE, relative_y // self.TILE_SIZE
        return grid_x, grid_y

    def _coords_absolute_to_cam_relative(self, x: int, y: int, info: dict) -> tuple[int, int]:
        cam_x, cam_y = self._get_camera_coords(info)
        relative_x = int(x) - cam_x
        relative_y = int(y) - cam_y
        return relative_x, relative_y

    def _get_camera_coords(self, info: dict):
        cam_x = info.get('cam_x', None)
        ram = None
        if cam_x is None:
            ram = self.env.unwrapped.get_ram()
            cam_x = self._combine_low_and_high_bits(ram[0x1A], ram[0x1B])
        cam_y = info.get('cam_y', None)
        if cam_y is None:
            if ram is None:
                ram = self.env.unwrapped.get_ram()
            cam_y = self._combine_low_and_high_bits(ram[0x1C], ram[0x1D])
        return int(cam_x), int(cam_y)

In [12]:
import cv2

class DebugVisualizer():
    def __init__(self, screen_rows: int = 14, screen_columns: int = 16, tile_size: int = 16, render_grid=False):
        self.screen_rows = screen_rows
        self.screen_columns = screen_columns
        self.tile_size = tile_size
        self.render_grid = render_grid
        self.GRID_COLOR = (80, 80, 80)
        self.img_width = self.screen_columns * self.tile_size
        self.img_height = self.screen_rows * self.tile_size

    def overlay(self, original_frame, observation):
        game_view = original_frame.copy()
        if self.render_grid:
            game_view = self._draw_grid(game_view)
        matrix_view = self._get_observation_image(observation)
        return np.hstack((game_view, matrix_view))

    def _get_observation_image(self, observation):
        matrix_img = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        if observation is not None:
            matrix_img = self._populate_objects(matrix_img, observation)
        if self.render_grid:
            matrix_img = self._draw_grid(matrix_img)
        return matrix_img

    def _populate_objects(self, matrix_img, observation):
        for row in range(self.screen_rows):
            for col in range(self.screen_columns):
                tile = observation[row][col]
                self._draw_tile(matrix_img, col, row, tile)
        return matrix_img

    def _draw_tile(self, img, x: int, y: int, tile: Tile) -> None:
        if tile["type"] == TileType.EMPTY:
            return
        color = self._choose_tile_color(tile)
        x_start = int(x * self.tile_size)
        y_start = int(y * self.tile_size)
        x_end = x_start + self.tile_size - 1
        y_end = y_start + self.tile_size - 1
        cv2.rectangle(img, (x_start, y_start), (x_end, y_end), color, -1)

    # def _choose_tile_color(self, tile: Tile) -> tuple[int, int, int]:
    #     if tile["type"] == TileType.EMPTY:
    #         return (0, 0, 0)
    #     if tile["type"] == TileType.MARIO:
    #         return (0, 255, 0)
    #     if tile["type"] == TileType.SPRITE or tile["type"] == TileType.EXTENDED_SPRITE:
    #         id = int(tile["id"])
    #         return (int((id * 40) % 256), int((id * 80) % 256), 100)
    #     if tile["type"] == TileType.TERRAIN:
    #         id = int(tile["id"])
    #         return (int((id * 10) % 256), int((id * 20) % 256), int((id * 30) % 256))
    #     return (255, 0, 0)

    def _choose_tile_color(self, tile: Tile) -> tuple[int, int, int]:
        if tile["type"] == TileType.EMPTY:
            return (0, 0, 0)

        # Base settings to ensure visibility
        t_type = tile["type"]
        t_id = int(tile["id"])
        h, s, v = 0, 200, 255 # Default (Red)

        # Map Types to specific Hue ranges (OpenCV Hue is 0-179)
        if t_type == TileType.MARIO:
            h, s, v = 60, 255, 255   # Bright Green

        elif t_type == TileType.TERRAIN:
            # Blue/Cyan range: Shift hue slightly by ID to distinguish tiles
            h = 100 + (t_id % 20)
            s = 150 + (t_id % 100)  # Variety in "vividness"

        elif t_type == TileType.BLOCK:
            # Orange/Yellow range
            h = 20 + (t_id % 15)
            s = 180 + (t_id % 75)

        elif t_type == TileType.SPRITE:
            # Purple/Magenta range
            h = 140 + (t_id % 20)
            v = 200 + (t_id % 55)   # Variety in brightness

        elif t_type == TileType.EXTENDED_SPRITE:
            # Deep Reds (as requested)
            h = 0 + (t_id % 10)
            s = 255

        else:
            # Unknown types - High-contrast Lime/Cyan
            h, s, v = 85, 255, 255

        # Convert the single HSV pixel to BGR for OpenCV
        hsv_pixel = np.uint8([[[h, s, v]]])
        bgr_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2RGB)[0][0]

        return (int(bgr_pixel[0]), int(bgr_pixel[1]), int(bgr_pixel[2]))

    def _draw_grid(self, img):
        h, w = img.shape[:2]
        for x in range(0, w + 1, self.tile_size):
            cv2.line(img, (x, 0), (x, h), self.GRID_COLOR, 1)
        for y in range(0, h + 1, self.tile_size):
            cv2.line(img, (0, y), (w, y), self.GRID_COLOR, 1)
        return img


class HUDRenderer:
    """Renders a text+sparkline+histogram panel to attach to the right of the matrix view.
    Pulls its state from attributes set by EmulatorWrapper."""

    WIDTH = 200
    BG = (18, 18, 22)               # dark charcoal
    TEXT = (220, 220, 220)
    DIM = (120, 120, 120)
    POS_COLOR = (80, 220, 80)       # green for positive reward
    NEG_COLOR = (80, 80, 230)        # red for negative reward (BGR)
    SEP = (50, 50, 55)              # divider color
    FONT = cv2.FONT_HERSHEY_SIMPLEX
    FONT_SCALE = 0.38
    LINE_H = 13
    PAD_X = 6
    POS_FLASH_FRAMES = 5
    NEG_FLASH_FRAMES = 10
    SPARK_HEIGHT = 24
    SPARK_WINDOW = 100             # number of recent rewards to show

    def __init__(self, height=224, action_names=None):
        self.height = height
        self.action_names = action_names or []

    def render(self, state: dict) -> np.ndarray:
        img = np.full((self.height, self.WIDTH, 3), self.BG, dtype=np.uint8)
        y = self.LINE_H + 2

        # --- Basic state ---
        step = state.get('step', 0)
        max_steps = state.get('max_steps', 0)
        y = self._put(img, f"STEP   {step:>4d}/{max_steps}", y)

        # Reward row with color flash
        step_reward = state.get('step_reward', 0.0)
        frames_since_event = state.get('frames_since_reward_event', 9999)
        last_event_sign = state.get('last_reward_event_sign', 0)
        reward_color = self._reward_color(frames_since_event, last_event_sign)
        y = self._put(img, f"REWARD  {step_reward:+6.2f}", y, color=reward_color)

        # --- Sparkline ---
        y = self._divider(img, y)
        history = state.get('reward_history', [])
        self._draw_sparkline(img, y, history)
        y += self.SPARK_HEIGHT + 4

        # --- Totals and action ---
        y = self._divider(img, y)
        total = state.get('total_reward', 0.0)
        y = self._put(img, f"TOTAL  {total:+7.2f}", y)
        action_name = state.get('action_name', '--')
        # truncate if too long
        if len(action_name) > 11:
            action_name = action_name[:11]
        y = self._put(img, f"ACT  {action_name}", y)

        # --- Counters (two lines, compact) ---
        y = self._divider(img, y)
        d = state.get('deaths', 0); ii = state.get('interactions', 0)
        ss = state.get('sprites_seen', 0); rr = state.get('room_id', 0)
        y = self._put(img, f"D:{d:<3d} I:{ii:<4d} S:{ss:<3d}", y)
        # --- Position / powerup / goal ---
        mx = state.get('mario_x', 0); my = state.get('mario_y', 0)
        y = self._put(img, f"xy:{mx:>5d},{my:<4d} R:{rr:<2d}", y)
        powerup_str = self._powerup_name(state.get('powerup', 0))
        goal_mark = 'Y' if state.get('goal_reached', False) else '-'
        y = self._put(img, f"POW:{powerup_str:<4} GOAL:{goal_mark}", y)

        # --- Action histogram ---
        action_counts = state.get('action_counts', None)
        if action_counts is not None and len(self.action_names) > 0 and y < self.height - self.LINE_H:
            y = self._divider(img, y)
            y = self._draw_action_hist(img, y, action_counts)

        return img

    def _put(self, img, text, y, color=None):
        if color is None:
            color = self.TEXT
        cv2.putText(img, text, (self.PAD_X, y), self.FONT, self.FONT_SCALE, color, 1, cv2.LINE_AA)
        return y + self.LINE_H

    def _divider(self, img, y):
        cv2.line(img, (self.PAD_X, y - 4), (self.WIDTH - self.PAD_X, y - 4), self.SEP, 1)
        return y + 2

    def _reward_color(self, frames_since, sign):
        if sign > 0 and frames_since <= self.POS_FLASH_FRAMES:
            alpha = 1.0 - (frames_since / self.POS_FLASH_FRAMES)
            return self._lerp_color(self.TEXT, self.POS_COLOR, alpha)
        if sign < 0 and frames_since <= self.NEG_FLASH_FRAMES:
            alpha = 1.0 - (frames_since / self.NEG_FLASH_FRAMES)
            return self._lerp_color(self.TEXT, self.NEG_COLOR, alpha)
        return self.TEXT

    def _lerp_color(self, a, b, alpha):
        return tuple(int(a[i] * (1 - alpha) + b[i] * alpha) for i in range(3))

    def _powerup_name(self, p):
        return {0: 'Sml', 1: 'Big', 2: 'Cape', 3: 'Fire'}.get(int(p), '?')

    def _draw_sparkline(self, img, y_top, history):
        x0 = self.PAD_X
        x1 = self.WIDTH - self.PAD_X
        w = x1 - x0
        h = self.SPARK_HEIGHT
        # baseline (zero-line)
        y_mid = y_top + h // 2
        cv2.line(img, (x0, y_mid), (x1, y_mid), self.SEP, 1)
        if len(history) < 2:
            return
        recent = history[-self.SPARK_WINDOW:]
        # normalize against max absolute value in window (avoid div-by-zero)
        amp = max((abs(v) for v in recent), default=1.0) or 1.0
        n = len(recent)
        # draw as little vertical bars from the midline (more readable than a polyline for sparse signals)
        for i, v in enumerate(recent):
            if v == 0:
                continue
            x = x0 + int(i * (w - 1) / max(n - 1, 1))
            bar_h = int((abs(v) / amp) * (h // 2 - 1))
            if v > 0:
                cv2.line(img, (x, y_mid - 1), (x, y_mid - 1 - bar_h), self.POS_COLOR, 1)
            else:
                cv2.line(img, (x, y_mid + 1), (x, y_mid + 1 + bar_h), self.NEG_COLOR, 1)

    # Short labels for the histogram so columns stay narrow and labels don't collide
    # when truncated. Falls back to the full action_name if not in this map.
    HIST_SHORT_LABELS = {
        'NOOP':       'NOOP',
        'LEFT':       'L',
        'RIGHT':      'R',
        'RIGHT+RUN':  'R+RUN',
        'RIGHT+JMP':  'R+JMP',
        'RIGHT+RJ':   'R+RJ',
        'JUMP':       'JUMP',
        'DOWN':       'DOWN',
        'LEFT+RUN':   'L+RUN',
        'LEFT+JMP':   'L+JMP',
        'LEFT+RJ':    'L+RJ',
    }

    def _draw_action_hist(self, img, y, counts):
        total = sum(counts) or 1
        # Header spans both columns
        cv2.putText(img, 'ACTIONS:', (self.PAD_X, y), self.FONT, self.FONT_SCALE, self.DIM, 1, cv2.LINE_AA)
        y += self.LINE_H - 3
        # Two-column layout: split actions in half
        n = len(self.action_names)
        if n == 0:
            return y
        col_split = (n + 1) // 2  # left col gets ceil(n/2)
        col_width = (self.WIDTH - self.PAD_X * 2) // 2  # ~94px per column
        label_w = 36  # space reserved for the label text
        bar_max_w = col_width - label_w - 2
        row_h = 10
        rows_per_col = max(col_split, n - col_split)
        # Drawing positions for the two columns
        col_x = [self.PAD_X, self.PAD_X + col_width]
        for r in range(rows_per_col):
            for col in (0, 1):
                idx = (col_split * col) + r if col == 1 else r
                # Left col is 0..col_split-1, right col is col_split..n-1
                idx = r if col == 0 else col_split + r
                if idx >= n or idx >= len(counts):
                    continue
                if y + r * row_h > self.height - 2:
                    continue
                name = self.action_names[idx]
                label = self.HIST_SHORT_LABELS.get(name, name[:5])
                frac = counts[idx] / total
                bar_w = int(frac * bar_max_w)
                lx = col_x[col]
                cy = y + r * row_h
                cv2.putText(img, label, (lx, cy), self.FONT, 0.32, self.DIM, 1, cv2.LINE_AA)
                if bar_w > 0:
                    cv2.rectangle(img, (lx + label_w, cy - 6), (lx + label_w + bar_w, cy - 1), self.TEXT, -1)
        return y + rows_per_col * row_h

In [13]:
from cv2.gapi import add
from typing import Any
import gymnasium as gym


class EmulatorWrapper(gym.Wrapper):
    # Reward constants — simplified two-parameter reward function
    ALPHA = 5.0   # reward per new sprite type discovered this episode
    BETA  = 10.0  # reward per adjacent sprite interaction (score delta > 0)
    GAMMA = 25.0  # bonus the first time Mario takes a previously-discovered sprite type

    def __init__(self, env, max_episode_length, render_debug=False, render_grid=False, logger=None):
        self.logger = logger if logger is not None else DummyLogger()
        self.env = env
        self._max_episode_length = max_episode_length
        self._world_parser = WorldParser(self.env, logger=logger)
        self._debug_visualizer = DebugVisualizer(render_grid=render_grid)
        self._hud = HUDRenderer(height=224)
        self._current_step = 0
        self.render_debug = render_debug
        self.render_grid = render_grid
        self.observation = None
        self._action_object_counts = {}  # key: (action, sprite_type) → count
        # Global across-episode tracking
        self.seen_ids = set()
        # Per-episode state
        self._episode_seen_sprite_types = set()
        self._episode_taken_sprite_types = set()  # sprite types Mario has touched/collected
        self._prev_sprite_slots = {}              # {slot: (type, tile_x, tile_y)} for disappearance detection
        self._episode_death_count = 0
        self._starting_lives = None
        self._prev_lives = None
        self._prev_score = None
        self._episode_total_reward = 0.0
        self._episode_interactions = 0
        self._prev_powerup = None
        # HUD tracking
        self._last_step_reward = 0.0
        self._last_info = {}
        self._last_discrete_action = None
        self._reward_history = []
        self._action_counts = None
        self._frames_since_reward_event = 9999
        self._last_reward_event_sign = 0
        super().__init__(self.env)
        # Override AFTER super().__init__ so it is not overwritten
        self.observation_space = gym.spaces.Box(
            low=0, high=65535, shape=(14, 16), dtype=np.float32
        )

    def reset(self, **kwargs):
        self._current_step = 0
        obs, info = self.env.reset(**kwargs)

        # Skip spawn animation — needs 150 frames to finish
        noop = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
        for _ in range(150):
            _, _, _, _, info = self.env.step(noop)
        obs, _, _, _, info = self.env.step(noop)

        matrix_obs = self._world_parser.get_screen_matrix_simplified(info)
        self.observation = self._world_parser.get_screen_matrix(info)
        self._action_object_counts = {}  # key: (action, sprite_type) → count

        # Reset per-episode state
        self._episode_seen_sprite_types = set()
        self._episode_taken_sprite_types = set()
        self._prev_sprite_slots = {}
        self._episode_death_count = 0
        self._starting_lives = int(info.get('lives', 4))
        self._prev_lives = self._starting_lives
        self._prev_score = int(info.get('score', 0))
        self._episode_total_reward = 0.0
        self._episode_interactions = 0
        self._prev_powerup = int(info.get('mario_powerup', 0))
        # Reset HUD tracking
        self._last_step_reward = 0.0
        self._last_info = dict(info)
        self._last_discrete_action = None
        self._reward_history = []
        if self._hud.action_names and self._action_counts is None:
            self._action_counts = [0] * len(self._hud.action_names)
        elif self._action_counts is not None:
            self._action_counts = [0] * len(self._action_counts)
        self._frames_since_reward_event = 9999
        self._last_reward_event_sign = 0
        self._has_found_ram_offset = False
        return matrix_obs.astype(np.float32), self._reformat_info(info)

    def render(self):
        original_frame = self.env.render()
        if not self.render_debug:
            return original_frame
        debug_frame = self._debug_visualizer.overlay(original_frame, self.observation)
        hud_state = self._build_hud_state()
        hud_img = self._hud.render(hud_state)
        dh, dw = debug_frame.shape[:2]
        hh, hw = hud_img.shape[:2]
        if hh != dh:
            hud_img = cv2.resize(hud_img, (hw, dh), interpolation=cv2.INTER_NEAREST)
        return np.hstack((debug_frame, hud_img))

    def _build_hud_state(self) -> dict:
        info = self._last_info or {}
        action_name = '--'
        if self._last_discrete_action is not None and self._hud.action_names:
            idx = int(self._last_discrete_action)
            if 0 <= idx < len(self._hud.action_names):
                action_name = self._hud.action_names[idx]
        return {
            'step': self._current_step,
            'max_steps': self._max_episode_length,
            'step_reward': self._last_step_reward,
            'total_reward': self._episode_total_reward,
            'action_name': action_name,
            'deaths': self._episode_death_count,
            'interactions': self._episode_interactions,
            'sprites_seen': len(self._episode_seen_sprite_types),
            'room_id': int(info.get('room_id', 0)),
            'mario_x': int(info.get('x', 0)),
            'mario_y': int(info.get('y', 0)),
            'powerup': int(info.get('mario_powerup', 0)),
            'goal_reached': int(info.get('end_level_timer', 0)) > 0,
            'reward_history': list(self._reward_history),
            'action_counts': list(self._action_counts) if self._action_counts is not None else None,
            'frames_since_reward_event': self._frames_since_reward_event,
            'last_reward_event_sign': self._last_reward_event_sign,
        }

    def step(self, action):
        _, _, _, _, info = self.env.step(action)
        self._current_step += 1
        self.observation = self._world_parser.get_screen_matrix(info)
        observation = self._world_parser.get_screen_matrix_simplified(info)
        self._update_seen_ids(observation, info)
        reward = self._compute_reward(info)  # no action parameter needed
        self._prev_powerup = int(info.get('mario_powerup', self._prev_powerup))
        self._episode_total_reward += reward
        terminated = self._has_died(info)
        truncated = self._has_reached_max_steps()
        # Update trackers AFTER reward computation
        self._prev_lives = int(info.get('lives', self._prev_lives))
        self._prev_score = int(info.get('score', self._prev_score))
        # HUD state update
        self._last_step_reward = reward
        self._last_info = dict(info)
        self._reward_history.append(reward)
        self._frames_since_reward_event += 1
        if reward > 0:
            self._frames_since_reward_event = 0
            self._last_reward_event_sign = 1
        elif reward < 0:
            self._frames_since_reward_event = 0
            self._last_reward_event_sign = -1
        if self._action_counts is not None and self._last_discrete_action is not None:
            idx = int(self._last_discrete_action)
            if 0 <= idx < len(self._action_counts):
                self._action_counts[idx] += 1
        new_info = self._reformat_info(info)
        if terminated or truncated:
            self.logger.info(
                f"Episode end: steps={self._current_step} "
                f"total_reward={self._episode_total_reward:.2f} "
                f"deaths={self._episode_death_count} "
                f"interactions={self._episode_interactions} "
                f"sprites_seen={len(self._episode_seen_sprite_types)}"
            )
        return observation.astype(np.float32), reward, terminated, truncated, new_info

    def _proximity_weight(self, mario_tx, mario_ty, sprite_tx, sprite_ty):
      """Weight by Manhattan distance — only sprites within 3 tiles count."""
      dist = abs(sprite_tx - mario_tx) + abs(sprite_ty - mario_ty)
      if dist == 1:   return 1.00  # adjacent
      elif dist == 2: return 0.50
      elif dist == 3: return 0.25
      else:           return 0.00  # too far


    def _compute_reward(self, info: dict) -> float:
      reward = 0.0

      # (1) Alpha — new sprite type seen this episode
      new_sprite_types = self._collect_new_sprite_types(info)
      reward += self.ALPHA * len(new_sprite_types)

      # (2) Beta — action-object combo, weighted by proximity, decaying with repetition
      mario_tx = int(info.get('x', 0)) // 16
      mario_ty = int(info.get('y', 0)) // 16

      discrete_action = self._last_discrete_action or 0

      # Snapshot current active sprites for disappearance detection later
      current_slots = {}
      for i in range(12):
          if info.get(f'sprite_status_{i}', 0) == 0:
              continue
          sprite_type = int(info.get(f'sprite_type_{i}', 0))
          sx = (info.get(f'sprite_x_high_{i}', 0) << 8) | info.get(f'sprite_x_low_{i}', 0)
          sy = (info.get(f'sprite_y_high_{i}', 0) << 8) | info.get(f'sprite_y_low_{i}', 0)
          sprite_tx = sx // 16
          sprite_ty = sy // 16
          current_slots[i] = (sprite_type, sprite_tx, sprite_ty)

          weight = self._proximity_weight(mario_tx, mario_ty, sprite_tx, sprite_ty)
          if weight == 0:
              continue

          combo = (int(discrete_action), sprite_type)
          count = self._action_object_counts.get(combo, 0) + 1
          self._action_object_counts[combo] = count

          combo_reward = self.BETA * weight / np.sqrt(count)
          reward += combo_reward

          if count == 1:
              self.logger.info(
                  f"New action-object combo at step {self._current_step}: "
                  f"action={discrete_action} sprite_type={sprite_type} "
                  f"weight={weight} reward={combo_reward:.2f}"
        )

      # (3) Gamma — first-time-taken bonus.
      # A sprite that was active last frame near Mario and is now gone counts as 'taken'.
      # If its type was already discovered (seen) earlier this episode but never yet taken,
      # give a one-shot bonus.
      for slot, (prev_type, prev_tx, prev_ty) in self._prev_sprite_slots.items():
          # Was the sprite alive last frame near Mario?
          prev_weight = self._proximity_weight(mario_tx, mario_ty, prev_tx, prev_ty)
          if prev_weight < 1.0:
              continue
          # Is the slot now empty, or holding a different sprite?
          cur_slot = current_slots.get(slot)
          disappeared = cur_slot is None or cur_slot[0] != prev_type
          if not disappeared:
              continue
          key = ('reg', prev_type)
          if key in self._episode_seen_sprite_types and key not in self._episode_taken_sprite_types:
              self._episode_taken_sprite_types.add(key)
              reward += self.GAMMA
              self._episode_interactions += 1
              self.logger.info(
                  f"[FIRST-TAKE bonus] step={self._current_step} sprite_type={prev_type} "
                  f"reward=+{self.GAMMA:.2f}"
              )

      # Update sprite snapshot for next step
      self._prev_sprite_slots = current_slots

      return reward

    def _collect_new_sprite_types(self, info: dict) -> set:
        new_types = set()
        for i in range(12):
            if info.get(f'sprite_status_{i}', 0) != 0:
                t = int(info.get(f'sprite_type_{i}', 0))
                key = ('reg', t)
                if key not in self._episode_seen_sprite_types:
                    self._episode_seen_sprite_types.add(key)
                    new_types.add(key)
        for i in range(10):
            t = int(info.get(f'ext_sprite_type_{i}', 0))
            if t != 0:
                key = ('ext', t)
                if key not in self._episode_seen_sprite_types:
                    self._episode_seen_sprite_types.add(key)
                    new_types.add(key)
        return new_types

    def _has_died(self, info) -> bool:
        if self._starting_lives is None:
            return False
        return int(info.get('lives', self._starting_lives)) < self._starting_lives

    def _has_reached_max_steps(self) -> bool:
        return self._current_step >= self._max_episode_length

    def _reformat_info(self, info) -> dict:
        cam_x, cam_y = self._world_parser._get_camera_coords(info)
        return {
            **info,
            'mario_x': int(info.get('x', 0)),
            'mario_y': int(info.get('y', 0)),
            'cam_x': cam_x,
            'cam_y': cam_y,
            'room_id': int(info.get('room_id', 0)),
            'goal_reached': int(info.get('end_level_timer', 0)) > 0,
            'episode_deaths': self._episode_death_count,
            'episode_interactions': self._episode_interactions,
            'episode_sprites_seen': len(self._episode_seen_sprite_types),
            'episode_total_reward': self._episode_total_reward,
            'global_tiles_seen': len(self.seen_ids),
        }

    def _update_seen_ids(self, observation, info) -> None:
        has_found_new_tiles = False
        rows = len(observation)
        cols = len(observation[0])
        for row in range(rows):
            for col in range(cols):
                tile_id = observation[row, col]
                if tile_id not in self.seen_ids:
                    self.logger.info(f"New tile id found: {tile_id:03x}")
                    self.seen_ids.add(tile_id)
                    has_found_new_tiles = True
        if has_found_new_tiles:
            self._debug_print_simplified_tile_ids(observation)

    def _debug_print_simplified_tile_ids(self, observation):
        self.logger.debug("--- Tile IDs ---")
        rows = 14
        cols = 16
        for y in range(rows):
            row_str = ""
            for x in range(cols):
                val = observation[y, x]
                row_str += f"{val:03x} "
            self.logger.debug(row_str)


In [14]:
WALK_RIGHT_ACTION = [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]

In [15]:
RUN_NAME = "smoke_test_v2"
LEVEL = "DonutPlains1"
MAX_STEPS = 3000
LOG_LEVEL = "INFO"

In [16]:
from gymnasium.wrappers import RecordVideo, FrameStackObservation, FlattenObservation

In [17]:
import time

import stable_retro as retro

logger = get_logger(RUN_NAME, LOG_LEVEL)
env = retro.make(game='SuperMarioWorld-Snes-v0', state=LEVEL, render_mode="rgb_array")
try:
    env = EmulatorWrapper(env, MAX_STEPS, render_debug=True, render_grid=True, logger=logger)
    env = RecordVideo(env, video_folder="./", name_prefix=RUN_NAME, episode_trigger=lambda x: True)
    obs = env.reset()
    done = False
    while not done:
        env.render()
        obs, reward, terminated, truncated, info = env.step(WALK_RIGHT_ACTION)
        # logger.debug(obs)
        done = terminated or truncated
        if done:
            logger.info(f"Terminated: {terminated}")
            logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    env.close()

2026-05-04 14:47:42 [INFO] Session log for run smoke_test_v2 with level [INFO] initialized at: smoke_test_v2_20260504_144742.log
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-04 14:47:42 [INFO] New tile id found: 000
2026-05-04 14:47:42 [INFO] New tile id found: 001
2026-05-04 14:47:42 [INFO] New tile id found: 373
2026-05-04 14:47:42 [INFO] New tile id found: 374
2026-05-04 14:47:42 [INFO] New tile id found: 379
2026-05-04 14:47:42 [INFO] New tile id found: 300
2026-05-04 14:47:42 [INFO] New tile id found: 33f
2026-05-04 14:47:43 [INFO] New tile id found: 301
2026-05-04 14:47:43 [INFO] New tile id found: 340
2026-05-04 14:47:44 [INFO] New tile id found: 173
2026-05-04 14:47:44 [INFO] New action-object combo at step 266: action=0 sprite_type=115 weight=0.25 reward=2.50
202

# Discrete action wrapper

Reduces the 12-bit MultiBinary SNES action space to 8 curated actions a human would plausibly use in DonutPlains 1. Crucially includes DOWN so the agent can crouch into pipes (secret exit requires this).

In [18]:
class SMWDiscretizer(gym.ActionWrapper):
    """Map Discrete(N) to the 12-bit SNES action vector.
    SNES button order (stable-retro): [B, Y, SELECT, START, UP, DOWN, LEFT, RIGHT, A, X, L, R]
    In SMW: B=jump, Y=run, A=spin jump."""

    # Named button indices for readability
    B, Y, SELECT, START, UP, DOWN, LEFT, RIGHT, A, X, L, R = range(12)

    ACTION_NAMES = [
        'NOOP', 'LEFT', 'RIGHT', 'RIGHT+RUN',
        'RIGHT+JMP', 'RIGHT+RJ', 'JUMP', 'DOWN',
        'LEFT+RUN', 'LEFT+JMP', 'LEFT+RJ',
    ]

    def __init__(self, env):
        super().__init__(env)
        combos = [
            [],                               # 0: NOOP
            [self.LEFT],                      # 1: LEFT
            [self.RIGHT],                     # 2: RIGHT
            [self.RIGHT, self.Y],             # 3: RIGHT + run
            [self.RIGHT, self.B],             # 4: RIGHT + jump
            [self.RIGHT, self.Y, self.B],     # 5: RIGHT + run + jump
            [self.B],                         # 6: jump in place
            [self.DOWN],                      # 7: crouch / enter pipe
            [self.LEFT, self.Y],              # 8: LEFT + run
            [self.LEFT, self.B],              # 9: LEFT + jump
            [self.LEFT, self.Y, self.B],     # 10: LEFT + run + jump
        ]
        self._action_map = []
        for combo in combos:
            vec = np.zeros(12, dtype=np.uint8)
            for idx in combo:
                vec[idx] = 1
            self._action_map.append(vec)
        self.action_space = gym.spaces.Discrete(len(self._action_map))
        # Wire action names + counts into EmulatorWrapper HUD if present
        inner = self._find_emulator_wrapper()
        if inner is not None:
            inner._hud.action_names = list(self.ACTION_NAMES)
            inner._action_counts = [0] * len(self.ACTION_NAMES)

    def _find_emulator_wrapper(self):
        e = self.env
        while e is not None:
            if isinstance(e, EmulatorWrapper):
                return e
            e = getattr(e, 'env', None)
        return None

    def action(self, action):
        # Stash the discrete action on the inner wrapper for HUD display
        inner = self._find_emulator_wrapper()
        if inner is not None:
            inner._last_discrete_action = int(action)
        return self._action_map[int(action)]

# Sanity check: random policy

Runs a few episodes with random actions to verify the pipeline end-to-end (observation shape, reward signal, info dict, video recording). No learning — we just want to see rewards move and Mario do stuff on screen.

In [23]:
def build_env(run_name, level, max_steps, log_level='INFO', record_video=True, discrete_actions=True):
    logger = get_logger(run_name, log_level)
    env = retro.make(game='SuperMarioWorld-Snes-v0', state=level, render_mode='rgb_array')
    env = EmulatorWrapper(env, max_steps, render_debug=True, render_grid=True, logger=logger)
    if discrete_actions:
        env = SMWDiscretizer(env)
    env = FrameStackObservation(env, 4)
    env = FlattenObservation(env)
    if record_video:
        env = RecordVideo(env, video_folder='./', name_prefix=run_name, episode_trigger=lambda x: True)
    return env, logger

N_RANDOM_EPISODES = 1
env, logger = build_env('random_sanity', LEVEL, MAX_STEPS, LOG_LEVEL)
try:
    for ep in range(N_RANDOM_EPISODES):
        obs, info = env.reset()
        done = False
        ep_reward = 0.0
        ep_steps = 0
        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            ep_steps += 1
            done = terminated or truncated
        logger.info(
            f'[random ep {ep}] steps={ep_steps} reward={ep_reward:.2f} '
            f"deaths={info.get('episode_deaths', 0)} interactions={info.get('episode_interactions', 0)} "
            f"sprites_seen={info.get('episode_sprites_seen', 0)} goal={info.get('goal_reached', False)}"
        )
finally:
    env.close()

2026-05-04 14:49:37 [INFO] Session log for run random_sanity with level [INFO] initialized at: random_sanity_20260504_144937.log
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-04 14:49:38 [INFO] New tile id found: 000
2026-05-04 14:49:38 [INFO] New tile id found: 001
2026-05-04 14:49:38 [INFO] New tile id found: 373
2026-05-04 14:49:38 [INFO] New tile id found: 374
2026-05-04 14:49:38 [INFO] New tile id found: 379
2026-05-04 14:49:38 [INFO] New tile id found: 300
2026-05-04 14:49:38 [INFO] New tile id found: 33f
2026-05-04 14:49:55 [INFO] Episode end: steps=3000 total_reward=0.00 deaths=0 interactions=0 sprites_seen=0
2026-05-04 14:49:55 [INFO] [random ep 0] steps=3000 reward=0.00 deaths=0 interactions=0 sprites_seen=0 goal=False


# PPO smoke test

Very short PPO run (20k timesteps ≈ 20 episodes) using MlpPolicy. **This is NOT expected to produce a smart agent** — it's a smoke test to confirm the training loop runs end-to-end, loss decreases, and tensorboard gets logs. Real training takes millions of steps.

If this cell runs without crashing and the `rollout/ep_rew_mean` line in tensorboard is not flat-zero, the pipeline works and we can scale up.

In [19]:
!pip install -q stable-baselines3 tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 44.4 MB/s eta 0:00:00


In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback
import glob, os

PPO_TIMESTEPS = 5_000_000
PPO_RUN_NAME  = 'ppo_left'
CHECKPOINT_DIR = '/content/drive/MyDrive/smw-checkpoints/'

train_env, train_logger = build_env(
    PPO_RUN_NAME + '_train', LEVEL, MAX_STEPS, LOG_LEVEL, record_video=False
)
train_env = Monitor(train_env)

# ── Load latest checkpoint ────────────────────────────────────────────────────
checkpoints = sorted([
    f for f in glob.glob(CHECKPOINT_DIR + f'{PPO_RUN_NAME}_*.zip')
    if 'final' not in f
])

if checkpoints:
    latest = checkpoints[-1]
    print(f"Resuming from: {latest}")
    model = PPO.load(latest, env=train_env, device='cpu')
    reset_timesteps = False
else:
    print("No checkpoint found — starting from scratch")
    model = PPO(
        'MlpPolicy', train_env, verbose=1,
        n_steps=512, batch_size=64, n_epochs=4,
        learning_rate=2.5e-4, gamma=0.99, ent_coef=0.05,
        tensorboard_log=CHECKPOINT_DIR + 'tb_logs/',
        policy_kwargs=dict(net_arch=[256, 256]),
        device='cpu',
    )
    reset_timesteps = True

print(f"Starting from timestep: {model.num_timesteps:,}")

# ── Checkpoint callback ───────────────────────────────────────────────────────
checkpoint_cb = CheckpointCallback(
    save_freq=100_000,
    save_path=CHECKPOINT_DIR,
    name_prefix=PPO_RUN_NAME,
    verbose=1,
)

try:
    model.learn(
        total_timesteps=PPO_TIMESTEPS,
        callback=checkpoint_cb,
        tb_log_name=PPO_RUN_NAME,
        progress_bar=True,
        reset_num_timesteps=reset_timesteps,
    )
    model.save(CHECKPOINT_DIR + f'{PPO_RUN_NAME}_final')
    train_logger.info("Training complete")
finally:
    train_env.close()

  81% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━ 4,030,619/5,000,000  [ 11:59:47 < 2:43:20 , 99 it/s ]

In [ ]:
import glob, os, gc
import numpy as np
from collections import Counter
from stable_baselines3 import PPO

# ── Configuration ─────────────────────────────────────────────────────────────
N_EVAL_EPISODES = 10
DETERMINISTIC   = False

# ── Find the latest checkpoint ────────────────────────────────────────────────
candidates = sorted(
    glob.glob(CHECKPOINT_DIR + f'{PPO_RUN_NAME}_*.zip'),
    key=os.path.getmtime
)
if not candidates:
    raise FileNotFoundError(f'No checkpoints found at {CHECKPOINT_DIR}{PPO_RUN_NAME}_*.zip')

latest_ckpt = candidates[-1]
print(f'Loading checkpoint: {latest_ckpt}')
print(f'  ({len(candidates)} checkpoints total)')

model = PPO.load(latest_ckpt, device='cpu')
print(f'Model loaded — trained for {model.num_timesteps:,} timesteps')

# ── Single env, reset between episodes ───────────────────────────────────────
# RecordVideo with episode_trigger=lambda x: True records every episode automatically
env, eval_logger = build_env(
    PPO_RUN_NAME + '_eval_latest', LEVEL, MAX_STEPS, LOG_LEVEL,
    record_video=True
)

results = []
try:
    for ep in range(N_EVAL_EPISODES):
        obs, info = env.reset()
        done = False
        ep_reward = 0.0
        ep_actions = []

        while not done:
            action, _ = model.predict(obs, deterministic=DETERMINISTIC)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
            ep_actions.append(int(action))
            done = terminated or truncated

        results.append({
            'episode':      ep + 1,
            'reward':       ep_reward,
            'deaths':       info.get('episode_deaths', 0),
            'interactions': info.get('episode_interactions', 0),
            'sprites_seen': info.get('episode_sprites_seen', 0),
            'goal':         info.get('goal_reached', False),
            'final_x':      info.get('mario_x', 0),
            'top_action':   Counter(ep_actions).most_common(1)[0][0] if ep_actions else -1,
            'steps':        len(ep_actions),
        })

        eval_logger.info(
            f"[ep {ep+1:2d}/{N_EVAL_EPISODES}] reward={ep_reward:7.2f} "
            f"steps={results[-1]['steps']:4d} "
            f"deaths={results[-1]['deaths']} "
            f"interactions={results[-1]['interactions']} "
            f"sprites={results[-1]['sprites_seen']} "
            f"final_x={results[-1]['final_x']:4d} "
            f"goal={results[-1]['goal']}"
        )
finally:
    env.close()
    gc.collect()

# ── Summary ───────────────────────────────────────────────────────────────────
rewards      = [r['reward']       for r in results]
deaths       = [r['deaths']       for r in results]
interactions = [r['interactions'] for r in results]
sprites      = [r['sprites_seen'] for r in results]
final_xs     = [r['final_x']      for r in results]
goals        = [r['goal']         for r in results]

ACTION_NAMES = [
    'NOOP', 'LEFT', 'RIGHT', 'RIGHT+RUN',
    'RIGHT+JMP', 'RIGHT+RJ', 'JUMP', 'DOWN',
    'LEFT+RUN', 'LEFT+JMP', 'LEFT+RJ',
]

print('\n' + '=' * 60)
print(f'  EVAL SUMMARY  —  {N_EVAL_EPISODES} episodes  —  deterministic={DETERMINISTIC}')
print(f'  Model: {os.path.basename(latest_ckpt)}')
print(f'  Trained for {model.num_timesteps:,} timesteps')
print('=' * 60)
print(f'  Reward         mean={np.mean(rewards):8.2f}  std={np.std(rewards):7.2f}  '
      f'min={np.min(rewards):7.2f}  max={np.max(rewards):7.2f}')
print(f'  Deaths         mean={np.mean(deaths):8.2f}  '
      f'min={int(np.min(deaths)):7d}  max={int(np.max(deaths)):7d}')
print(f'  Interactions   mean={np.mean(interactions):8.2f}  '
      f'min={int(np.min(interactions)):7d}  max={int(np.max(interactions)):7d}')
print(f'  Sprites seen   mean={np.mean(sprites):8.2f}  '
      f'min={int(np.min(sprites)):7d}  max={int(np.max(sprites)):7d}')
print(f'  Final x-pos    mean={np.mean(final_xs):8.0f}  '
      f'min={int(np.min(final_xs)):7d}  max={int(np.max(final_xs)):7d}')
print(f'  Goal reached   {sum(goals)}/{N_EVAL_EPISODES} '
      f'({100*sum(goals)/N_EVAL_EPISODES:.0f}%)')
print('-' * 60)
top_actions = Counter(r['top_action'] for r in results)
print('  Most-used action per episode:')
for action_idx, count in top_actions.most_common():
    name = ACTION_NAMES[action_idx] if 0 <= action_idx < len(ACTION_NAMES) else f'?{action_idx}'
    print(f'    {name:12s} in {count:2d} episodes')
print('=' * 60)

2026-05-04 14:54:09 [INFO] Session log for run ppo_left_eval_latest with level [INFO] initialized at: ppo_left_eval_latest_20260504_145409.log


Loading checkpoint: /content/drive/MyDrive/smw-checkpoints/ppo_left_4000000_steps.zip
  (40 checkpoints total)
Model loaded — trained for 4,000,000 timesteps


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-04 14:54:10 [INFO] New tile id found: 000
2026-05-04 14:54:10 [INFO] New tile id found: 001
2026-05-04 14:54:10 [INFO] New tile id found: 373
2026-05-04 14:54:10 [INFO] New tile id found: 374
2026-05-04 14:54:10 [INFO] New tile id found: 379
2026-05-04 14:54:10 [INFO] New tile id found: 300
2026-05-04 14:54:10 [INFO] New tile id found: 33f
2026-05-04 14:54:19 [INFO] New tile id found: 301
2026-05-04 14:54:19 [INFO] New tile id found: 340
2026-05-04 14:54:20 [INFO] New tile id found: 173
2026-05-04 14:54:20 [INFO] New action-object combo at step 1403: action=8 sprite_type=115 weight=0.25 reward=2.50
2026-05-04 14:54:20 [INFO] New action-object combo at step 1404: action=5 sprite_type=115 weight=0.25 reward=2.50
2026-05-04 14:54: